In [1]:
import json
from pathlib import Path

import pandas as pd

RAW_DIR = Path("data/raw")

# Load raw users JSON saved by extract.py
with open(RAW_DIR / "users_raw.json", encoding="utf-8") as f:
    users_raw = json.load(f)

# Flatten nested fields (e.g. address -> city becomes address_city)
users_df = pd.json_normalize(users_raw, sep="_")

print("Shape:", users_df.shape)
print(users_df.columns.tolist())

Shape: (208, 52)
['id', 'firstName', 'lastName', 'maidenName', 'age', 'gender', 'email', 'phone', 'username', 'password', 'birthDate', 'image', 'bloodGroup', 'height', 'weight', 'eyeColor', 'ip', 'macAddress', 'university', 'ein', 'ssn', 'userAgent', 'role', 'hair_color', 'hair_type', 'address_address', 'address_city', 'address_state', 'address_stateCode', 'address_postalCode', 'address_coordinates_lat', 'address_coordinates_lng', 'address_country', 'bank_cardExpire', 'bank_cardNumber', 'bank_cardType', 'bank_currency', 'bank_iban', 'company_department', 'company_name', 'company_title', 'company_address_address', 'company_address_city', 'company_address_state', 'company_address_stateCode', 'company_address_postalCode', 'company_address_coordinates_lat', 'company_address_coordinates_lng', 'company_address_country', 'crypto_coin', 'crypto_wallet', 'crypto_network']


In [2]:
# Whitelist approach: keep only analytics-relevant columns and rename them to snake_case
USER_COLUMNS = {
    "id": "user_id",
    "firstName": "first_name",
    "lastName": "last_name",
    "age": "age",
    "gender": "gender",
    "role": "role",
    "address_city": "city",
    "address_state": "state",
    "address_stateCode": "state_code",
    "address_postalCode": "postal_code",
    "address_country": "country",
    "company_department": "company_department",
    "company_title": "job_title",
}

users_clean = users_df[list(USER_COLUMNS)].rename(columns=USER_COLUMNS)

print("Shape:", users_clean.shape)
users_clean.info()
users_clean.head()

Shape: (208, 13)
<class 'pandas.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   user_id             208 non-null    int64
 1   first_name          208 non-null    str  
 2   last_name           208 non-null    str  
 3   age                 208 non-null    int64
 4   gender              208 non-null    str  
 5   role                208 non-null    str  
 6   city                208 non-null    str  
 7   state               208 non-null    str  
 8   state_code          208 non-null    str  
 9   postal_code         208 non-null    str  
 10  country             208 non-null    str  
 11  company_department  208 non-null    str  
 12  job_title           208 non-null    str  
dtypes: int64(2), str(11)
memory usage: 39.0 KB


,user_id,first_name,last_name,age,gender,role,city,state,state_code,postal_code,country,company_department,job_title
0,1,Emily,Johnson,29,female,admin,Phoenix,Mississippi,MS,29112,United States,Engineering,Sales Manager
1,2,Michael,Williams,36,male,admin,Houston,Alabama,AL,38807,United States,Support,Support Specialist
2,3,Sophia,Brown,43,female,admin,Washington,Alabama,AL,32822,United States,Research and Development,Accountant
3,4,James,Davis,46,male,admin,Seattle,Pennsylvania,PA,68354,United States,Support,Research Analyst
4,5,Emma,Miller,31,female,admin,Jacksonville,Colorado,CO,26593,United States,Human Resources,Quality Assurance Engineer


In [3]:
# Load raw products JSON saved by extract.py
with open(RAW_DIR / "products_raw.json", encoding="utf-8") as f:
    products_raw = json.load(f)

# Flatten nested dictionaries (dimensions, meta); list fields stay as-is
products_df = pd.json_normalize(products_raw, sep="_")

print("Shape:", products_df.shape)
print(products_df.columns.tolist())

# Show only columns that contain missing values
nulls = products_df.isnull().sum()
print("\nColumns with missing values:")
print(nulls[nulls > 0])

Shape: (194, 27)
['id', 'title', 'description', 'category', 'price', 'discountPercentage', 'rating', 'stock', 'tags', 'brand', 'sku', 'weight', 'warrantyInformation', 'shippingInformation', 'availabilityStatus', 'reviews', 'returnPolicy', 'minimumOrderQuantity', 'images', 'thumbnail', 'dimensions_width', 'dimensions_height', 'dimensions_depth', 'meta_createdAt', 'meta_updatedAt', 'meta_barcode', 'meta_qrCode']

Columns with missing values:
brand    92
dtype: int64


In [4]:
# Check which categories have missing brands (before filling)
print("Categories with missing brand:")
print(products_df[products_df["brand"].isnull()]["category"].value_counts())

# Whitelist approach: keep analytics-relevant columns and rename to snake_case
PRODUCT_COLUMNS = {
    "id": "product_id",
    "title": "product_name",
    "category": "category",
    "brand": "brand",
    "sku": "sku",
    "price": "price",
    "discountPercentage": "discount_percentage",
    "rating": "rating",
    "stock": "stock",
    "weight": "weight",
    "availabilityStatus": "availability_status",
    "minimumOrderQuantity": "minimum_order_quantity",
    "warrantyInformation": "warranty_information",
    "shippingInformation": "shipping_information",
    "returnPolicy": "return_policy",
    "meta_createdAt": "created_at",
}

products_clean = products_df[list(PRODUCT_COLUMNS)].rename(columns=PRODUCT_COLUMNS)

# Products without a brand are generic items, so label them explicitly
products_clean["brand"] = products_clean["brand"].fillna("Unbranded")

# Convert ISO timestamp text to a real datetime type
products_clean["created_at"] = pd.to_datetime(products_clean["created_at"])

print("\nShape:", products_clean.shape)
products_clean.info()

Categories with missing brand:
category
kitchen-accessories    30
groceries              27
sports-accessories     17
home-decoration         5
tops                    5
womens-dresses          5
womens-jewellery        3
Name: count, dtype: int64

Shape: (194, 16)
<class 'pandas.DataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype              
---  ------                  --------------  -----              
 0   product_id              194 non-null    int64              
 1   product_name            194 non-null    str                
 2   category                194 non-null    str                
 3   brand                   194 non-null    str                
 4   sku                     194 non-null    str                
 5   price                   194 non-null    float64            
 6   discount_percentage     194 non-null    float64            
 7   rating                  194 non-null    float64   

In [5]:
# Explode the nested reviews list: one row per review, linked back to its product
reviews_df = pd.json_normalize(
    products_raw,
    record_path="reviews",   # the list to turn into rows
    meta=["id"],             # parent field to carry into each row
)

# Whitelist + snake_case (reviewerEmail dropped as PII)
REVIEW_COLUMNS = {
    "id": "product_id",
    "rating": "review_rating",
    "comment": "review_comment",
    "date": "review_date",
    "reviewerName": "reviewer_name",
}

reviews_clean = reviews_df[list(REVIEW_COLUMNS)].rename(columns=REVIEW_COLUMNS)
reviews_clean["review_date"] = pd.to_datetime(reviews_clean["review_date"])

# Surrogate key: a unique ID for each review (the source has none)
reviews_clean.insert(0, "review_id", range(1, len(reviews_clean) + 1))

print("Shape:", reviews_clean.shape)
print("Reviews per product (min/max):",
      reviews_clean.groupby("product_id").size().min(), "/",
      reviews_clean.groupby("product_id").size().max())
reviews_clean.head()

Shape: (582, 6)
Reviews per product (min/max): 3 / 3


,review_id,product_id,review_rating,review_comment,review_date,reviewer_name
0,1,1,3,Would not recommend!,2025-04-30 09:41:02.053000+00:00,Eleanor Collins
1,2,1,4,Very satisfied!,2025-04-30 09:41:02.053000+00:00,Lucas Gordon
2,3,1,5,Highly impressed!,2025-04-30 09:41:02.053000+00:00,Eleanor Collins
3,4,2,5,Great product!,2025-04-30 09:41:02.053000+00:00,Savannah Gomez
4,5,2,4,Awesome product!,2025-04-30 09:41:02.053000+00:00,Christian Perez


In [6]:
# Load raw carts JSON saved by extract.py
with open(RAW_DIR / "carts_raw.json", encoding="utf-8") as f:
    carts_raw = json.load(f)

print("Number of carts:", len(carts_raw))

# Cart-level fields (one value per cart)
print("\nCart-level keys:", list(carts_raw[0].keys()))

# Product-level fields (one value per product inside a cart)
print("Product-level keys:", list(carts_raw[0]["products"][0].keys()))

# How many products each cart holds -> total tells us the expected row count after exploding
products_per_cart = pd.Series([len(cart["products"]) for cart in carts_raw])
print("\nProducts per cart (min/max):", products_per_cart.min(), "/", products_per_cart.max())
print("Expected rows after explode:", products_per_cart.sum())

Number of carts: 208

Cart-level keys: ['id', 'products', 'total', 'discountedTotal', 'userId', 'totalProducts', 'totalQuantity']
Product-level keys: ['id', 'title', 'price', 'quantity', 'total', 'discountPercentage', 'discountedTotal', 'thumbnail']

Products per cart (min/max): 2 / 6
Expected rows after explode: 800


In [7]:
# Explode the nested products list: one row per product per cart
cart_items_df = pd.json_normalize(
    carts_raw,
    record_path="products",        # the list to turn into rows
    meta=["id", "userId"],         # cart-level fields to carry into each row
    meta_prefix="cart_",           # avoids clash: cart "id" vs product "id"
)

# Whitelist + snake_case (title dropped: already in products_clean; thumbnail not needed)
CART_ITEM_COLUMNS = {
    "cart_id": "cart_id",
    "cart_userId": "user_id",
    "id": "product_id",
    "price": "price",
    "quantity": "quantity",
    "total": "total",
    "discountPercentage": "discount_percentage",
    "discountedTotal": "discounted_total",
}

cart_items_clean = cart_items_df[list(CART_ITEM_COLUMNS)].rename(columns=CART_ITEM_COLUMNS)

# Validation checks
print("Shape:", cart_items_clean.shape)
print("Nulls:", cart_items_clean.isnull().sum().sum())
print("Duplicate (cart_id, product_id):", cart_items_clean.duplicated(["cart_id", "product_id"]).sum())

# Referential integrity: every product/user must exist in its clean dimension table
print("Unknown product_ids:", (~cart_items_clean["product_id"].isin(products_clean["product_id"])).sum())
print("Unknown user_ids:", (~cart_items_clean["user_id"].isin(users_clean["user_id"])).sum())

cart_items_clean.info()
cart_items_clean.head()

Shape: (800, 8)
Nulls: 0
Duplicate (cart_id, product_id): 12
Unknown product_ids: 0
Unknown user_ids: 0
<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   cart_id              800 non-null    object 
 1   user_id              800 non-null    object 
 2   product_id           800 non-null    int64  
 3   price                800 non-null    float64
 4   quantity             800 non-null    int64  
 5   total                800 non-null    float64
 6   discount_percentage  800 non-null    float64
 7   discounted_total     800 non-null    float64
dtypes: float64(4), int64(2), object(2)
memory usage: 50.1+ KB


,cart_id,user_id,product_id,price,quantity,total,discount_percentage,discounted_total
0,1,1,162,29.99,4,119.96,12.13,105.41
1,1,1,113,3999.99,3,11999.97,12.10,10547.97
2,1,1,122,299.99,3,899.97,6.69,839.76
3,1,1,138,8.99,2,17.98,1.71,17.67
4,2,2,86,19.99,5,99.95,6.83,93.12


In [8]:
# Cart header table: one row per cart (products list excluded, already exploded)
CART_COLUMNS = {
    "id": "cart_id",
    "userId": "user_id",
    "total": "total",
    "discountedTotal": "discounted_total",
    "totalProducts": "total_products",
    "totalQuantity": "total_quantity",
}

carts_df = pd.json_normalize(carts_raw)
carts_clean = carts_df[list(CART_COLUMNS)].rename(columns=CART_COLUMNS)

# Meta columns from json_normalize can arrive as generic "object" type; align ID types for joining
cart_items_clean[["cart_id", "user_id"]] = cart_items_clean[["cart_id", "user_id"]].astype("int64")

# Recompute cart totals from the line items
items_agg = cart_items_clean.groupby("cart_id").agg(
    items_total=("total", "sum"),
    items_discounted_total=("discounted_total", "sum"),
    items_count=("product_id", "count"),
    items_quantity=("quantity", "sum"),
).reset_index()

recon = carts_clean.merge(items_agg, on="cart_id", how="left")

# Reconciliation: header values must equal the sum of their line items (0.01 tolerance for float rounding)
print("Shape:", carts_clean.shape)
print("Nulls:", carts_clean.isnull().sum().sum())
print("Total mismatches:", ((recon["total"] - recon["items_total"]).abs() > 0.01).sum())
print("Discounted total mismatches:", ((recon["discounted_total"] - recon["items_discounted_total"]).abs() > 0.01).sum())
print("Product count mismatches:", (recon["total_products"] != recon["items_count"]).sum())
print("Quantity mismatches:", (recon["total_quantity"] != recon["items_quantity"]).sum())

carts_clean.head()

Shape: (208, 6)
Nulls: 0
Total mismatches: 0
Discounted total mismatches: 0
Product count mismatches: 0
Quantity mismatches: 0


,cart_id,user_id,total,discounted_total,total_products,total_quantity
0,1,1,13037.88,11510.81,4,12
1,2,2,139.93,125.70,2,7
2,3,3,1794.85,1625.77,6,15
3,4,4,689.93,636.81,2,7
4,5,5,1467.88,1205.80,3,12


In [9]:
# Kaggle files live in a subfolder of data/raw
KAGGLE_DIR = RAW_DIR / "kaggle"

# Load the main orders file (one row per order)
sales_df = pd.read_csv(KAGGLE_DIR / "ecommerce_sales_customer_analytics_150k.csv")

print("Shape:", sales_df.shape)

# Show every column's type next to one sample value (46 columns -> easier to read vertically)
overview = pd.DataFrame({
    "dtype": sales_df.dtypes.astype(str),
    "sample_value": sales_df.iloc[0],
})
print(overview.to_string())

Shape: (138116, 46)
                            dtype                  sample_value
order_id                      str                    ORD-301242
order_date                    str                    2023-11-06
order_time                    str                      16:37:47
order_status                  str                     Completed
sales_channel                 str                    Mobile App
customer_id                   str                   CUST-003102
customer_name                 str                  Jasmine Ryan
customer_age                int64                            55
gender                        str                          Male
customer_segment              str                      Consumer
customer_type                 str                         Loyal
customer_city                 str              Lake Williamberg
customer_state                str                         Texas
customer_country              str                           USA
region              

In [10]:
# Show only the middle columns that were truncated in the previous output
print(overview.iloc[23:43].to_string())

                            dtype                  sample_value
delivery_status               str                       On Time
return_status                 str                           NaN
return_reason                 str                           NaN
customer_rating           float64                           3.5
review_sentiment              str                      Positive
customer_review               str  Satisfied with the purchase.
marketing_channel             str                        Direct
campaign_name                 str              Default_Campaign
coupon_code                   str                           NaN
loyalty_points_earned       int64                            94
loyalty_points_redeemed     int64                            44
quantity                    int64                             5
gross_sales               float64                       1350.19
discount_amount           float64                        477.89
tax_amount                float64       

In [11]:
# Reload with postal code as text, so any leading zeros in the file are preserved
sales_df = pd.read_csv(
    KAGGLE_DIR / "ecommerce_sales_customer_analytics_150k.csv",
    dtype={"customer_postal_code": str},
)

# Combine date + time into one full timestamp (strict format: fails loudly on bad values)
sales_df["order_datetime"] = pd.to_datetime(
    sales_df["order_date"] + " " + sales_df["order_time"],
    format="%Y-%m-%d %H:%M:%S",
)

# Keep a pure date column for daily/monthly reporting; the time text is now redundant
sales_df["order_date"] = pd.to_datetime(sales_df["order_date"], format="%Y-%m-%d")
sales_df = sales_df.drop(columns="order_time")

# Checks
print("Shape:", sales_df.shape)
print(sales_df[["order_date", "order_datetime", "customer_postal_code"]].dtypes)
print("Date range:", sales_df["order_date"].min().date(), "to", sales_df["order_date"].max().date())
print("Postal code lengths:", sales_df["customer_postal_code"].str.len().value_counts().to_dict())

Shape: (138116, 46)
order_date              datetime64[us]
order_datetime          datetime64[us]
customer_postal_code               str
dtype: object
Date range: 2021-01-01 to 2025-12-31
Postal code lengths: {5: 138116}


In [12]:
# 1) All campaign values, with NULL counted as its own group
print("campaign_name values:")
print(sales_df["campaign_name"].value_counts(dropna=False))

# 2) Which marketing channels do NULL / Default_Campaign orders come from?
print("\nCampaign vs marketing channel:")
print(pd.crosstab(
    sales_df["campaign_name"].fillna("<NULL>"),
    sales_df["marketing_channel"],
))

# 3) Coupon values, with NULL counted as its own group
print("\ncoupon_code values:")
print(sales_df["coupon_code"].value_counts(dropna=False))

campaign_name values:
campaign_name
NaN                   83233
Default_Campaign      22989
Google_Search_Q1       2748
Google_Shopping        2746
Google_Display         2731
FB_Retargeting         2270
FB_Dynamic             2233
Friend_Invite          2232
Referral_Program       2198
FB_Prospecting         2161
Insta_Influencer       1822
Insta_Reels            1799
Insta_Stories          1773
Abandoned_Cart         1520
Weekly_Newsletter      1459
Promo_Email            1459
Influencer_Program     1404
Affiliate_Partner      1339
Name: count, dtype: int64

Campaign vs marketing channel:
marketing_channel   Affiliate  Direct  Email Marketing  Facebook Ads  \
campaign_name                                                          
<NULL>                   4105   12333             6780          9987   
Abandoned_Cart              0       0             1520             0   
Affiliate_Partner        1339       0                0             0   
Default_Campaign            0    8182     

In [13]:
# Focus only on the two "no real campaign" candidates: NULL and Default_Campaign
no_campaign = sales_df[
    sales_df["campaign_name"].isna() | (sales_df["campaign_name"] == "Default_Campaign")
]

print("Marketing channel for NULL vs Default_Campaign orders:")
print(pd.crosstab(
    no_campaign["marketing_channel"],
    no_campaign["campaign_name"].fillna("<NULL>"),
))

# Coupon: only the NULL count is needed here
print("\ncoupon_code NULLs:", sales_df["coupon_code"].isna().sum())
print("Distinct coupon codes:", sales_df["coupon_code"].nunique())

Marketing channel for NULL vs Default_Campaign orders:
campaign_name      <NULL>  Default_Campaign
marketing_channel                          
Affiliate            4105                 0
Direct              12333              8182
Email Marketing      6780                 0
Facebook Ads         9987                 0
Google Ads          12408                 0
Instagram            8407                 0
Organic Search      16733             11009
Referral             6761                 0
TikTok               3236              2201
YouTube              2483              1597

coupon_code NULLs: 110502
Distinct coupon codes: 40


In [14]:
# campaign_name NULLs appear across all channels, including paid ones (Google/Facebook Ads),
# so they mean "campaign not tracked", not "organic" -> label as Unattributed
sales_df["campaign_name"] = sales_df["campaign_name"].fillna("Unattributed")

# coupon_code NULL simply means the customer used no coupon
sales_df["coupon_code"] = sales_df["coupon_code"].fillna("No Coupon")

# Remaining NULLs should now be only the business-meaningful ones
remaining_nulls = sales_df.isnull().sum()
print(remaining_nulls[remaining_nulls > 0])

delivery_days               24557
estimated_delivery_days     24557
return_status              128654
return_reason              128654
customer_rating             24557
review_sentiment            24557
customer_review             24557
dtype: int64


In [15]:
print("Shape:", sales_df.shape)
print("Date range:", sales_df["order_date"].min().date(), "to", sales_df["order_date"].max().date())

Shape: (138116, 46)
Date range: 2021-01-01 to 2025-12-31


In [16]:
# The three remaining Kaggle tables (already known to have no nulls/duplicates)
KAGGLE_FILES = {
    "customers": "customer_master.csv",
    "products": "product_catalog.csv",
    "order_items": "order_items.csv",
}

kaggle_dfs = {}
for name, file_name in KAGGLE_FILES.items():
    df = pd.read_csv(KAGGLE_DIR / file_name)
    kaggle_dfs[name] = df

    # Same overview as before: type + one sample value per column
    print(f"===== {name} {df.shape} =====")
    print(pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "sample_value": df.iloc[0],
    }).to_string())
    print()

===== customers (25000, 11) =====
                             dtype  sample_value
customer_id                    str   CUST-000001
customer_name                  str  Donna Miller
customer_age                 int64            54
gender                         str        Female
customer_segment               str      Consumer
customer_city                  str    Laurenland
customer_state                 str         Dubai
customer_country               str           UAE
region                         str         South
customer_postal_code         int64         11253
customer_acquisition_cost  float64         13.58

===== products (1175, 9) =====
                       dtype                                     sample_value
product_id               str                                      PROD-000001
product_name             str  Apple Sharable bifurcated algorithm Smartphones
product_category         str                                      Electronics
product_subcategory      str      

In [17]:
# Reload customers with postal code as text (preserve any leading zeros)
customers_kaggle = pd.read_csv(
    KAGGLE_DIR / "customer_master.csv",
    dtype={"customer_postal_code": str},
)

# Kaggle discount is a 0-1 fraction; DummyJSON's discount_percentage is 0-100.
# Rename to avoid mixing two scales under one name.
order_items_kaggle = kaggle_dfs["order_items"].rename(
    columns={"discount_percentage": "discount_rate"}
)

# Checks
print("customers postal code dtype:", customers_kaggle["customer_postal_code"].dtype)
print("Postal code lengths:", customers_kaggle["customer_postal_code"].str.len().value_counts().to_dict())
print("discount_rate min/max:", order_items_kaggle["discount_rate"].min(), "/", order_items_kaggle["discount_rate"].max())

customers postal code dtype: str
Postal code lengths: {5: 22624, 4: 2252, 3: 124}
discount_rate min/max: 0.0 / 0.6


In [18]:
postal_len = customers_kaggle["customer_postal_code"].str.len()

# 1) Are short codes tied to specific countries?
print("Postal code length by country:")
print(pd.crosstab(customers_kaggle["customer_country"], postal_len))

# 2) Do the same customers have a different (5-digit) code in the sales file?
sales_codes = sales_df[["customer_id", "customer_postal_code"]].drop_duplicates("customer_id")
short_codes = customers_kaggle.loc[postal_len < 5, ["customer_id", "customer_country", "customer_postal_code"]]

comparison = short_codes.merge(sales_codes, on="customer_id", how="inner", suffixes=("_master", "_sales"))
print("\nShort-code customers also found in sales file:", len(comparison))
print(comparison.head(10).to_string())

Postal code length by country:
customer_postal_code   3     4      5
customer_country                     
Australia              6   106   1121
Canada                 7   124   1148
Germany                7   183   1856
India                  2    91    948
UAE                    3    56    716
UK                    19   324   3358
USA                   80  1368  13477

Short-code customers also found in sales file: 2366
   customer_id customer_country customer_postal_code_master customer_postal_code_sales
0  CUST-000015              USA                        4028                      04028
1  CUST-000028              USA                        6357                      06357
2  CUST-000041               UK                        1757                      01757
3  CUST-000048            India                        4725                      04725
4  CUST-000053              USA                        8271                      08271
5  CUST-000061          Germany                     

In [19]:
# Source file stored postal codes as numbers, dropping leading zeros -> restore to 5 characters
customers_kaggle["customer_postal_code"] = customers_kaggle["customer_postal_code"].str.zfill(5)

# Check 1: every code should now be 5 characters
print("Postal code lengths:", customers_kaggle["customer_postal_code"].str.len().value_counts().to_dict())

# Check 2: repaired codes should now match the sales file for the same customers
recheck = customers_kaggle[["customer_id", "customer_postal_code"]].merge(
    sales_codes, on="customer_id", how="inner", suffixes=("_master", "_sales")
)
mismatches = (recheck["customer_postal_code_master"] != recheck["customer_postal_code_sales"]).sum()
print("Customers compared:", len(recheck))
print("Postal code mismatches vs sales file:", mismatches)

Postal code lengths: {5: 25000}
Customers compared: 24911
Postal code mismatches vs sales file: 0


In [20]:
import sys
sys.path.append("src")  # let the notebook import our script as a module

from transform import transform_web_cart_items

# 1) Result from the script's own function
script_items = transform_web_cart_items(carts_raw)
script_dups = script_items[script_items.duplicated(["cart_id", "product_id"], keep=False)]
print("Script  -> rows:", len(script_items), "| duplicate rows:", len(script_dups))

# 2) Result from the notebook variable (Step 3.4b)
nb_dups = cart_items_clean[cart_items_clean.duplicated(["cart_id", "product_id"], keep=False)]
print("Notebook-> rows:", len(cart_items_clean), "| duplicate rows:", len(nb_dups))

# Show the duplicated rows side by side
print(script_dups.sort_values(["cart_id", "product_id"]).to_string())

Script  -> rows: 800 | duplicate rows: 24
Notebook-> rows: 800 | duplicate rows: 24
     cart_item_id  cart_id  user_id  product_id    price  quantity     total  discount_percentage  discounted_total
21             22        7        7          56    49.99         1     49.99                14.04             42.97
25             26        7        7          56    49.99         1     49.99                14.04             42.97
141           142       38       38         110    14.99         2     29.98                19.40             24.16
143           144       38       38         110    14.99         2     29.98                19.40             24.16
254           255       69       69         122   299.99         2    599.98                 6.69            559.84
255           256       69       69         122   299.99         5   1499.95                 6.69           1399.60
332           333       89       89         150    29.99         2     59.98                19.51       

In [21]:
PROCESSED_DIR = Path("data/processed")

# Read two saved files back and confirm the types survived the round trip
orders_check = pd.read_parquet(PROCESSED_DIR / "erp_orders.parquet")
customers_check = pd.read_parquet(PROCESSED_DIR / "erp_customers.parquet")

print(orders_check[["order_date", "order_datetime", "customer_postal_code", "is_repeat_customer"]].dtypes)
print("\nSample postal codes (should keep leading zeros):")
print(customers_check.loc[customers_check["customer_id"].isin(["CUST-000015", "CUST-000066"]),
                          ["customer_id", "customer_postal_code"]].to_string())

order_date              datetime64[us]
order_datetime          datetime64[us]
customer_postal_code               str
is_repeat_customer                bool
dtype: object

Sample postal codes (should keep leading zeros):
    customer_id customer_postal_code
14  CUST-000015                04028
65  CUST-000066                00643
